# E-Commerce Sales Data Cleaning Pipeline

**Objective:** To clean and standardize a dataset of messy e-commerce transaction records for reliable business analytics.

**Portfolio Focus:** Python, Pandas, Data Cleaning Pipelines, Regex Pattern Matching, Data Type Optimization.

In [17]:
import pandas as pd
import numpy as np
import re

# Define local paths relative to the notebook's location
input_filename = input('Enter your raw filename (e.g., ecommerce_sales_raw.csv): ')
output_filename = "cleaned_" + input_filename

### Raw Data Analysis & Identified Issues

* **Test/Sandbox Data:** Several rows contain 'test' or 'sandbox' in the `order_notes`. These are non-revenue generating records that must be purged before analysis.

* **Missing Data:** The columns `customer_email`, `sku`, and `order_notes` contain missing values. `sku` nulls will be safely filled with `"SKU-000"` to preserve filter integrity and naming conventions.

* **Inconsistent Categorical Text:** `currency` and `status` have mixed cases (e.g., 'usd' vs 'USD') causing grouping fragmentation.

* **Messy SKU Formats:** Product codes lack a uniform structure (e.g., 'sku324' vs 'SKU-721').

* **Mixed Date Formats & Typing:** `order_date` contains multiple regional formats (European, US, text-based) and is currently stored as a string. These will be standardized to ISO 8601 and explicitly cast to `datetime64`.

* **Duplicate Records:** Duplicate `order_id`s exist. Redundant rows must be dropped while prioritizing the retention of rows with valuable `order_notes`.

In [25]:
# Ingestion: Automatically handling trailing spaces after delimiters
df = pd.read_csv(input_filename, skipinitialspace=True)

# Quick inspection of the raw data shape and initial rows
print(f"Initial shape: {df.shape}")
df.head()

Initial shape: (325, 8)


,order_id,customer_email,sku,amount,currency,order_date,order_notes,status
0,ORD00166,user131@example.com,SKU-721,761.08,usd,2023-06-25,sandbox,COMPLETED
1,ORD00097,user182@example.com,sku324,62.26,Eur,2022-03-03,NaN,COMPLETED
2,ORD00106,user122@example.com,SKU-735,1702.65,EUR,14/8/2024,urgent,pending
3,ORD00054,user95@example.com,sku873,367.77,EUR,Sep-14-2022,NaN,COMPLETED
4,ORD00095,user29@example.com,sku775,428.18,GBP,3/6/2024,real,COMPLETED


In [19]:
# 1. Remove Test/Sandbox Orders
test_keywords = ['test', 'sandbox']
mask = df['order_notes'].str.lower().str.contains('|'.join(test_keywords), na=False)
df = df[~mask].copy()

# 2. Handle Missing Values (Imputation)
df['customer_email'] = df['customer_email'].fillna("unknown@example.com")
df['sku'] = df['sku'].fillna("SKU-000") # Consistent placeholder for filter integrity
df['order_notes'] = df['order_notes'].fillna("")

# 3. Standardize Categorical Text Data
df['currency'] = df['currency'].astype(str).str.upper()
df['status'] = df['status'].astype(str).str.upper()

# 4. Standardize SKU Formatting using Regex
def clean_sku(sku):
    sku = str(sku).upper().strip()
    # Force 'SKU-XXXX' format for consistency
    return re.sub(r'^SKU-?(\d+)', r'SKU-\1', sku)

df['sku'] = df['sku'].apply(clean_sku)

# 5. Standardize Dates & Convert to Datetime object
df['order_date'] = pd.to_datetime(
    df['order_date'], 
    format='mixed', 
    dayfirst=True, 
    errors='coerce'
)

# 6. Handle Duplicates
# Sort to keep rows with notes first, then drop duplicate order IDs
df = df.sort_values(by=['order_id', 'order_notes'], ascending=[True, False])
df = df.drop_duplicates(subset=['order_id'], keep='first')

# Reset the index after all the dropping/sorting
df = df.reset_index(drop=True)

# Final audit
essential_cols = ['order_id', 'customer_email', 'sku', 'amount', 'currency', 'order_date', 'status']
print(f"Remaining missing values in essential columns: {df[essential_cols].isnull().sum().sum()}")
print(f"Final dataset shape: {df.shape}")
print(f"Order Date dtype: {df['order_date'].dtype}")

Remaining missing values in essential columns: 0
Final dataset shape: (224, 8)
Order Date dtype: datetime64[us]


In [20]:
# Export the finalized, clean dataframe
# Note: dates will automatically be exported in standard YYYY-MM-DD format by pandas
df.to_csv(output_filename, index=False)
print(f"Data cleaning complete. Cleaned file saved securely locally at: {output_filename}")

Data cleaning complete. Cleaned file saved securely locally at: cleaned_ecommerce_sales_raw.csv
